# EV Charging Lane - Find Best Locations

We look at roads and pick the best 4 spots for EV charging lanes based on EV traffic, safety, and road quality.

In [ ]:
%%sql -r ctx
USE DATABASE DATA5035;
USE SCHEMA SPRING26;

In [ ]:
# import tools that we need
import snowflake.snowpark as snowpark
from snowflake.snowpark.context import get_active_session
import pandas as pd

In [ ]:
# Now we connected to snowflake
session = get_active_session()

In [ ]:
# load 4 tables as we need
road_segments = session.table("DATA5035.SPRING26.ROAD_SEGMENTS").to_pandas()
traffic_counts = session.table("DATA5035.SPRING26.TRAFFIC_COUNTS").to_pandas()
incidents = session.table("DATA5035.SPRING26.INCIDENTS").to_pandas()
weather_risk = session.table("DATA5035.SPRING26.WEATHER_RISK").to_pandas()

print(f"Road Segments: {len(road_segments)} rows")
print(f"Traffic Counts: {len(traffic_counts)} rows")
print(f"Incidents: {len(incidents)} rows")
print(f"Weather Risk: {len(weather_risk)} rows")

In [ ]:
# merge all 4 tables on SEGMENT ID
df = road_segments.merge(traffic_counts, on='SEGMENT_ID', how='left')
df = df.merge(incidents, on='SEGMENT_ID', how='left')
df = df.merge(weather_risk, on='SEGMENT_ID', how='left')
print(f"Combined: {len(df)} rows")
df.head()

In [ ]:
# score each road 0-100
# demand = more EVs = higher
# safety = fewer crashes (60%) + lower weather risk 40% = higher
# road = more lanes + higher speed = higher
df['DEMAND_SCORE'] = (df['AADT_EV'] - df['AADT_EV'].min()) / (df['AADT_EV'].max() - df['AADT_EV'].min()) * 100

crash_norm = (df['CRASH_RATE'] - df['CRASH_RATE'].min()) / (df['CRASH_RATE'].max() - df['CRASH_RATE'].min())
weather_norm = (df['RISK_SCORE'] - df['RISK_SCORE'].min()) / (df['RISK_SCORE'].max() - df['RISK_SCORE'].min())
df['SAFETY_SCORE'] = 100 - (crash_norm * 60 + weather_norm * 40)

df['ROAD_SCORE'] = (df['LANES'] / df['LANES'].max() * 50) + (df['SPEED_LIMIT'] / df['SPEED_LIMIT'].max() * 50)

df['DEMAND_SCORE'] = df['DEMAND_SCORE'].round(0).astype(int)
df['SAFETY_SCORE'] = df['SAFETY_SCORE'].round(0).astype(int)
df['ROAD_SCORE'] = df['ROAD_SCORE'].round(0).astype(int)

def highlight_high(val):
    if isinstance(val, (int, float)) and val >= 70:
        return 'background-color: #90EE90'
    return ''

show_cols = ['SEGMENT_ID', 'INTERSTATE', 'DEMAND_SCORE', 'SAFETY_SCORE', 'ROAD_SCORE']
df[show_cols].head(10).style.map(highlight_high, subset=['DEMAND_SCORE', 'SAFETY_SCORE', 'ROAD_SCORE'])

In [ ]:
# weighted final score: demand 40%, safety 35%, road 25%
df['COMPOSITE_SCORE'] = (
    df['DEMAND_SCORE'] * 0.40 +
    df['SAFETY_SCORE'] * 0.35 +
    df['ROAD_SCORE'] * 0.25
).round(0).astype(int)

df = df.sort_values('COMPOSITE_SCORE', ascending=False)

def highlight_high(val):
    if isinstance(val, (int, float)) and val >= 70:
        return 'background-color: #90EE90'
    return ''

show_cols = ['SEGMENT_ID', 'INTERSTATE', 'DEMAND_SCORE', 'SAFETY_SCORE', 'ROAD_SCORE', 'COMPOSITE_SCORE']
df[show_cols].head(10).style.map(highlight_high, subset=['DEMAND_SCORE', 'SAFETY_SCORE', 'ROAD_SCORE', 'COMPOSITE_SCORE'])

In [ ]:
# top 4 from different highways so chargers are spread out
# the loop is flexible - it can pick 2 from one highway if needed
# NOTE: this is different from the SQL version which strictly picks 1 per highway and i will explain in REFLECTION
selected = []
corridors_used = set()

for idx, row in df.iterrows():
    corridor = row['INTERSTATE']
    if len(selected) < 4:
        if corridor not in corridors_used or len(corridors_used) >= 2:
            selected.append(row)
            corridors_used.add(corridor)
    if len(selected) >= 4:
        break

top_4 = pd.DataFrame(selected)
print("TOP 4 LOCATIONS:")

def highlight_row(row):
    return ['background-color: #90EE90'] * len(row)

top_4[['SEGMENT_ID', 'INTERSTATE', 'START_MILE', 'END_MILE', 'COMPOSITE_SCORE']].style.apply(highlight_row, axis=1)

In [ ]:
# Full results table is showing all segments with their scores.
# The 4 selected locations from above are highlighted in green.

output_cols = ['SEGMENT_ID', 'INTERSTATE', 'START_MILE', 'END_MILE', 
               'AADT_EV', 'CRASH_RATE', 'LANES', 'SPEED_LIMIT',
               'DEMAND_SCORE', 'SAFETY_SCORE', 'ROAD_SCORE', 'COMPOSITE_SCORE']

top_4_ids = top_4['SEGMENT_ID'].tolist()

def highlight_top4(row):
    if row['SEGMENT_ID'] in top_4_ids:
        return ['background-color: #90EE90'] * len(row)
    return [''] * len(row)

df[output_cols].style.apply(highlight_top4, axis=1)

In [ ]:
# Export all segments with scores to SEGMENTS.csv
# The file is available for download in the week07/ folder of this workspace
output_cols = ['SEGMENT_ID', 'INTERSTATE', 'START_MILE', 'END_MILE',
               'AADT_EV', 'CRASH_RATE', 'LANES', 'SPEED_LIMIT',
               'DEMAND_SCORE', 'SAFETY_SCORE', 'ROAD_SCORE', 'COMPOSITE_SCORE']

result_df = df[output_cols].copy()

top_4_ids = top_4['SEGMENT_ID'].tolist()
result_df['RECOMMENDED'] = result_df['SEGMENT_ID'].apply(lambda x: 'YES' if x in top_4_ids else 'NO')

result_df.to_csv('/tmp/SEGMENTS.csv', index=False)

print(f"Exported {len(result_df)} segments to SEGMENTS.csv")
print(f"NOTE: SEGMENTS.csv is available for download in the week07/ folder")
print(f"\nRecommended locations:")
result_df[result_df['RECOMMENDED'] == 'YES'][['SEGMENT_ID', 'INTERSTATE', 'COMPOSITE_SCORE', 'RECOMMENDED']]